In [ ]:
# --- Importaciones ---------------------------------------------------
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm

from sklearn.datasets import load_digits, load_breast_cancer
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score

import sklearn
print(f'numpy:   {np.__version__}')
print(f'sklearn: {sklearn.__version__}')

In [ ]:
# --- Cargar load_digits (sin internet, preinstalado) ----------------
digits = load_digits()           # 1797 imágenes 8x8 = 64 features
X_dig = digits.data              # shape: (1797, 64)
y_dig = digits.target            # etiquetas 0-9

semilla = 42
print(f'Shape: {X_dig.shape}')
print(f'Clases: {np.unique(y_dig)}')
# Salida esperada:
# Shape: (1797, 64)
# Clases: [0 1 2 3 4 5 6 7 8 9]

# Escalar SIEMPRE antes de PCA
scaler = StandardScaler()
X_sc = scaler.fit_transform(X_dig)

In [ ]:
# --- PCA a 2D -------------------------------------------------------
pca2 = PCA(n_components=2, random_state=semilla)
X_pca2 = pca2.fit_transform(X_sc)

var_exp = pca2.explained_variance_ratio_
print(f'PC1: {var_exp[0]:.2%} | PC2: {var_exp[1]:.2%}',
    f'| Total: {var_exp.sum():.2%}')
# Salida esperada:
# PC1: 12.04% | PC2: 9.67% | Total: 21.71%

# --- Visualización --------------------------------------------------
fig, ax = plt.subplots(figsize=(9, 6))
scatter = ax.scatter(
    X_pca2[:, 0], X_pca2[:, 1],
    c=y_dig, cmap='tab10', s=15, alpha=0.7
)
plt.colorbar(scatter, ax=ax, label='Dígito')
ax.set_xlabel(f'PC1 ({var_exp[0]:.1%})')
ax.set_ylabel(f'PC2 ({var_exp[1]:.1%})')
ax.set_title('PCA 2D — Dígitos MNIST (load_digits)')
plt.tight_layout()
plt.show()


In [ ]:
pca_full = PCA(random_state=semilla)   # todas las componentes
pca_full.fit(X_sc)

var_acum = np.cumsum(pca_full.explained_variance_ratio_)

# ¿Cuántos componentes para superar el 95%?
n_95 = np.argmax(var_acum >= 0.95) + 1
n_99 = np.argmax(var_acum >= 0.99) + 1
print(f'Componentes para 95% varianza: {n_95}')
print(f'Componentes para 99% varianza: {n_99}')
# Salida esperada:
# Componentes para 95% varianza: 29
# Componentes para 99% varianza: 43

plt.figure(figsize=(8, 4))
plt.plot(range(1, len(var_acum)+1), var_acum, 'o-', ms=3)
plt.axhline(0.95, color='red', linestyle='--', label='95%')
plt.axhline(0.99, color='orange', linestyle='--', label='99%')
plt.axvline(n_95, color='red', linestyle=':', alpha=0.5)
plt.xlabel('Número de componentes')
plt.ylabel('Varianza explicada acumulada')
plt.title('Varianza explicada acumulada — load_digits')
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# --- Cargas (loadings) de las primeras 2 componentes ----------------
cargas = pca2.components_         # shape: (2, 64)

# Mostrar los 5 píxeles con mayor carga en PC1
carga_pc1 = pd.Series(
    np.abs(cargas[0]),
    index=[f'pixel_{i}' for i in range(64)],
).sort_values(ascending=False)

print('Top 5 features en PC1 (por magnitud):')
print(carga_pc1.head())

# Visualizar las cargas de PC1 y PC2 como imágenes 8x8
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(8, 3))
ax1.imshow(cargas[0].reshape(8, 8), cmap='RdBu_r')
ax1.set_title('Cargas PC1')
ax1.axis('off')
ax2.imshow(cargas[1].reshape(8, 8), cmap='RdBu_r')
ax2.set_title('Cargas PC2')
ax2.axis('off')
plt.suptitle('Mapa de cargas — qué píxeles contribuyen a cada PC')
plt.tight_layout()
plt.show()

In [ ]:
# --- Comparar clasificadores con y sin PCA --------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X_sc, y_dig, test_size=0.2, random_state=semilla, stratify=y_dig
)

# Sin PCA
lr_base = LogisticRegression(max_iter=1000, random_state=semilla)
lr_base.fit(X_train, y_train)
acc_base = accuracy_score(y_test, lr_base.predict(X_test))
print(f'Sin PCA (64 features):    {acc_base:.4f}')

# Con PCA al 95% de varianza
pipeline_pca = Pipeline([
    ('pca', PCA(n_components=0.95, random_state=semilla)),
    ('clf', LogisticRegression(max_iter=1000, random_state=semilla)),
])
pipeline_pca.fit(X_train, y_train)
acc_pca = accuracy_score(y_test, pipeline_pca.predict(X_test))
n_comp = pipeline_pca.named_steps['pca'].n_components_
print(f'Con PCA ({n_comp} componentes): {acc_pca:.4f}')


In [ ]:
# --- Reconstruir imágenes desde distintos n_components --------------
n_list = [2, 5, 10, 20, 40, 64]
idx_ejemplo = 5   # índice del dígito a reconstruir
imagen_orig = X_sc[idx_ejemplo]

fig, ejes = plt.subplots(1, len(n_list)+1, figsize=(14, 2))
ejes[0].imshow(imagen_orig.reshape(8, 8), cmap='gray_r')
ejes[0].set_title(f'Original\n(64)')
ejes[0].axis('off')

for i, n in enumerate(n_list):
    pca_n = PCA(n_components=n, random_state=semilla)
    pca_n.fit(X_sc)
    # Proyectar y reconstruir
    proyectado = pca_n.transform(imagen_orig.reshape(1, -1))
    reconstruido = pca_n.inverse_transform(proyectado)
    var = pca_n.explained_variance_ratio_.sum()
    ejes[i+1].imshow(reconstruido.reshape(8, 8), cmap='gray_r')
    ejes[i+1].set_title(f'n={n}\n({var:.0%})')
    ejes[i+1].axis('off')

plt.suptitle('Reconstrucción desde PCA — pérdida de información', y=1.05)
plt.tight_layout()
plt.show()

In [ ]:
# --- t-SNE sobre load_digits ----------------------------------------
# Aplicar PCA primero para acelerar t-SNE (práctica estándar)
pca_50 = PCA(n_components=50, random_state=semilla)
X_pca50 = pca_50.fit_transform(X_sc)

tsne = TSNE(
    n_components=2,
    perplexity=30,
    max_iter=1000,
    random_state=semilla,
    learning_rate='auto',
    init='pca',           # inicialización con PCA (recomendado)
)
X_tsne = tsne.fit_transform(X_pca50)

fig, ax = plt.subplots(figsize=(9, 6))
scatter = ax.scatter(
    X_tsne[:, 0], X_tsne[:, 1],
    c=y_dig, cmap='tab10', s=15, alpha=0.8
)
plt.colorbar(scatter, ax=ax, label='Dígito')
ax.set_title('t-SNE 2D — Dígitos MNIST (load_digits)')
ax.set_xlabel('t-SNE 1')
ax.set_ylabel('t-SNE 2')
plt.tight_layout()
plt.show()

In [ ]:
# --- Comparar distintos valores de perplexity -----------------------
valores_perp = [5, 15, 30, 50]
fig, ejes = plt.subplots(1, 4, figsize=(16, 4))

for ax, perp in zip(ejes, valores_perp):
    tsne_p = TSNE(
        n_components=2, perplexity=perp,
        max_iter=1000, random_state=semilla,
        learning_rate='auto', init='pca',
    )
    X_p = tsne_p.fit_transform(X_pca50)
    ax.scatter(X_p[:, 0], X_p[:, 1], c=y_dig,
            cmap='tab10', s=10, alpha=0.7)
    ax.set_title(f'perplexity={perp}')
    ax.axis('off')

plt.suptitle('Efecto de perplexity en t-SNE')
plt.tight_layout()
plt.show()

In [ ]:
# --- Pipeline PCA -> t-SNE (recomendado para datasets grandes) ------
# Nota: TSNE no implementa la API transform() de sklearn,
# por lo que no puede ir dentro de un Pipeline estándar.
# El patrón correcto es aplicarlos en secuencia:

def reducir_visualizar(X, y, n_pca=50, perp=30, titulo=''):
    """Aplica PCA + t-SNE y grafica el resultado."""
    pca = PCA(n_components=n_pca, random_state=semilla)
    X_reducido = pca.fit_transform(X)
    tsne = TSNE(
        n_components=2, perplexity=perp,
        max_iter=1000, random_state=semilla,
        learning_rate='auto', init='pca',
    )
    X_2d = tsne.fit_transform(X_reducido)
    plt.figure(figsize=(8, 5))
    scatter = plt.scatter(
        X_2d[:, 0], X_2d[:, 1],
        c=y, cmap='tab10', s=15, alpha=0.8,
    )
    plt.colorbar(scatter)
    plt.title(titulo)
    plt.tight_layout()
    plt.show()

reducir_visualizar(
    X_sc, y_dig,
    n_pca=50, perp=30,
    titulo='PCA(50) + t-SNE(perp=30) — load_digits',
)

In [ ]:
# --- Breast Cancer: accuracy con distintos n_components -------------
bc = load_breast_cancer()
X_bc = StandardScaler().fit_transform(bc.data)
y_bc = bc.target

X_tr_bc, X_te_bc, y_tr_bc, y_te_bc = train_test_split(
    X_bc, y_bc, test_size=0.2,
    random_state=semilla, stratify=y_bc,
)

resultados = []
n_max = X_bc.shape[1]   # 30 features

for n in [2, 5, 10, 15, 20, 25, n_max]:
    pipe = Pipeline([
        ('pca', PCA(n_components=n, random_state=semilla)),
        ('clf', LogisticRegression(max_iter=1000, random_state=semilla)),
    ])
    pipe.fit(X_tr_bc, y_tr_bc)
    acc = accuracy_score(y_te_bc, pipe.predict(X_te_bc))
    var = pipe.named_steps['pca'].explained_variance_ratio_.sum()
    resultados.append({'n': n, 'acc': acc, 'var': var})
    print(f'n={n:2d} | var={var:.1%} | acc={acc:.4f}')
